<h1><center>Language translation (English to Spanish)</center></h1>

In [1]:
# Import libraries
from pathlib import Path
import torch
import torch.nn as nn

from src.data.data_loader import create_dataloaders
from src.model.transformer import build_transformer
from src.model.transformer import Transformer
from src.train.training import train_model
from src.utils.utils import get_device
from nltk.tokenize import word_tokenize
from pathlib import Path

from src.utils.constants import PADDING_ID, UNKNOWN_ID, START_OF_SENTENCE_ID, END_OF_SENTENCE_ID
from src.utils.constants import PADDING_VALUE, UNKNOWN_VALUE, START_OF_SENTENCE_VALUE, END_OF_SENTENCE_VALUE

import os
os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = '0.0'

In [5]:
# Initialize model and training parameters

translation_file = 'data/en-fr.csv'

# Size of embedding vector
d_model = 512
# Number of words in a source vocabulary
src_vocab_size = 14500
# Number of words in a target vocabulary
tgt_vocab_size = 29300
# Max sequence length for input words/tokens
src_seq_len = 75
# Max sequence length for output words/tokens
tgt_seq_len = 75
# Dropout rate
dropout = 0.1
# number of encoder blocks
num_layers = 4
# number of attention heads
num_heads = 8
# Number of hidden nodes for feed-forward layer
d_ff = 4*d_model

# Number of epochs
epochs = 10
# Batch size for training
batch_size = 512

In [6]:
# Get a device to use for training/inference
device = get_device()

# Create training and validation data loaders
train_dataloader, val_dataloader, src_word_to_id, tgt_word_to_id = create_dataloaders(translation_file,
                                                                                      batch_size,
                                                                                      src_seq_len, 
                                                                                      tgt_seq_len, 
                                                                                      src_vocab_size,
                                                                                      tgt_vocab_size)

In [1]:
# Download model from here and save it to the models directory (Since Github doesn't store large files)
# Download link: https://drive.google.com/file/d/1qftDIziMJyzCUC19jmcYdRDWcWLJXGVE/view?usp=sharing

In [11]:
# Create new instance of model and load saved state dict
MODEL_PATH = Path("models")
MODEL_NAME = "08_language_translation.pth"
MODEL_SAVE_PATH = MODEL_PATH / MODEL_NAME

loaded_model = build_transformer(d_model, src_vocab_size, tgt_vocab_size, src_seq_len, tgt_seq_len, 
                                 dropout, num_layers, num_heads, d_ff)
loaded_model.load_state_dict(torch.load(MODEL_SAVE_PATH))
loaded_model.to(device)

Transformer(
  (src_embed): Embedding(
    (embedding): Embedding(14500, 512)
  )
  (tgt_embed): Embedding(
    (embedding): Embedding(29300, 512)
  )
  (src_pos): PositionalEncoding(
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (tgt_pos): PositionalEncoding(
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): Encoder(
    (layers): ModuleList(
      (0-3): 4 x EncoderBlock(
        (self_attention): MultiHeadAttention(
          (dropout): Dropout(p=0.1, inplace=False)
          (query_linear_layer): Linear(in_features=512, out_features=512, bias=True)
          (key_linear_layer): Linear(in_features=512, out_features=512, bias=True)
          (value_linear_layer): Linear(in_features=512, out_features=512, bias=True)
          (output_linear_layer): Linear(in_features=512, out_features=512, bias=True)
        )
        (feed_forward): FeedForward(
          (linear_1): Linear(in_features=512, out_features=2048, bias=True)
          (dropout): Dropout(p=0.1, inplace=Fal

In [21]:
# Translate from English to French
def translate(input, max_tokens_to_generate):
    with torch.inference_mode():
        
        src_mask = (input == 0).view(input.size(0), 1, 1, input.size(-1))
        encoder_output = loaded_model.encode(input, src_mask)
        
        output = torch.tensor([START_OF_SENTENCE_VALUE], dtype=torch.long).unsqueeze(0).to(device)
        
        for _ in range(max_tokens_to_generate):
            
            decoder_output = loaded_model.decode(output, encoder_output, src_mask)
            y_logits = loaded_model.project(decoder_output)
            
            # for all the batches, get the embeds for last predicted sequence
            y_logits = y_logits[:, -1, :]
            
#             # for all the batches, get the embeds for last predicted sequence
#             probs = y_logits.softmax(dim=1)            
#             # get the probable token based on the input probs
#             idx_next = torch.multinomial(probs, num_samples=1)
            
            idx_next = torch.argmax(y_logits, dim=1).unsqueeze(0)

            output = torch.cat([output, idx_next], dim=1)
            
            if idx_next == END_OF_SENTENCE_VALUE:
                break
            
        return output

sentences = [
    ['I know that I\'m crazy.', 'Je sais que je suis fou.'],
    ['I suggest we don\'t even try.', 'Je suggère que nous n\'essayions même pas.'],
    ['He often sits by me and listens to music.', 'Il s\'assied souvent à côté de moi et écoute de la musique.']
]

sorted_items = sorted(tgt_word_to_id.items(), key=lambda item: item[1])
sorted_keys = [item[0] for item in sorted_items]

for sentence in sentences:
    
    sentence[0] = sentence[0].lower()
    
    print(f'Source: {sentence[0]}')
    print(f'Target: {sentence[1]}')
          
    input = [src_word_to_id.get(token, src_word_to_id.get(UNKNOWN_ID)) for token in word_tokenize(sentence[0])]
    input_tensor = torch.tensor([START_OF_SENTENCE_VALUE] + input + [END_OF_SENTENCE_VALUE], 
                                dtype=torch.long).unsqueeze(0).to(device)
    
    output_tensor = translate(input_tensor, input_tensor.size(1) + 5).squeeze()[1:-1]
    output_array_tokens = output_tensor.cpu().numpy()
    
    output_array_words = [sorted_keys[token] for token in output_array_tokens]
    print(f"Predicted: {' '.join(output_array_words)}\n")

Source: i know that i'm crazy.
Target: Je sais que je suis fou.
Predicted: je sais que je suis fou .

Source: i suggest we don't even try.
Target: Je suggère que nous n'essayions même pas.
Predicted: je suggère que nous ne essayons pas même d'essayer .

Source: he often sits by me and listens to music.
Target: Il s'assied souvent à côté de moi et écoute de la musique.
Predicted: il est souvent à peu près de moi et écoute de la musique .

